<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎵 HeartMuLa 3B - Standalone Music Generator</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Colab Free Tier (T4 GPU) - Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>Lyrics-to-Music Generation with Native FP16 Tensor Core Acceleration</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-T4%20GPU-4285F4?style=for-the-badge&logo=google-colab&logoColor=white" />
  <img src="https://img.shields.io/badge/Engine-FP16%20Tensor%20Cores-success?style=for-the-badge" />
  <br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy" target="_blank">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

---

### 🌟 Key Features & Optimizations
- ⚡ **Fast Setup (< 30s):** Targeted dependencies; removed slow `flash-attn` compilation.
- 🚀 **Dual FP16 Tensor Cores:** Both HeartMuLa 3B and HeartCodec run in native FP16 on Tesla T4 for 5x faster generation and instant audio decoding.
- ⚡ **Zero-Disk RAM Offload:** Models swap between CPU RAM and GPU in 0.5s instead of reading 44s from disk.
- 🔊 **Full Audio Fidelity:** HeartCodec weight alignment guarantees 100% weights load with zero silent audio.
- 🎛️ **Fast Draft Mode (CFG = 1.0):** Cuts token generation time in half for rapid preview (~45s for 15s audio, ~80s for 30s audio).
- 🎧 **Interactive Gradio Studio:** Real-time smooth frame progress and 3-button control (Generate, Stop, Clear).

---

In [ ]:
# @title 🚀 1. Setup Environment & Fast Dependencies
# @markdown Run this cell once to configure the environment and install dependencies in ~25s.
import os
import sys
import subprocess

# Prevent PyTorch memory fragmentation on Tesla T4
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("=" * 65)
print("🎵 HeartMuLa 3B - Standalone Music Generator")
print("Google Colab T4 GPU Edition - Created by AIQUEST Academy")
print("=" * 65)

# Step 1: Verify Hardware Accelerator
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    if "T4" in gpu_name:
        print("⚡ NVIDIA Tesla T4 detected: Native FP16 Tensor Core acceleration enabled!")
else:
    print("⚠️ WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")

# Step 2: Clone HeartMuLa repository
HEARTLIB_DIR = "/content/heartlib"
if not os.path.exists(HEARTLIB_DIR):
    print("\n[1/3] 📥 Cloning HeartMuLa repository...")
    subprocess.run(
        ["git", "clone", "https://github.com/HeartMuLa/heartlib.git", HEARTLIB_DIR, "-q"],
        check=True,
    )
    print("✓ Repository cloned successfully")
else:
    print("\n[1/3] ✓ Repository already cloned")

# Step 3: High-speed targeted dependency installation
print("\n[2/3] ⚡ Installing optimized runtime dependencies...")
# Note: flash-attn is omitted as FlashAttention does not support Turing sm_75 (T4)
# and heartlib uses native PyTorch SDPA (Cutlass Memory-Efficient Attention).
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "torchtune==0.4.0",
        "torchao==0.9.0",
        "vector_quantize_pytorch>=1.14.0",
        "huggingface_hub>=0.23.0",
        "soundfile>=0.12.1",
        "gradio>=4.40.0",
        "safetensors>=0.4.0",
        "psutil",
    ],
    check=True,
)
print("✓ Core dependencies installed")

# Step 4: Install heartlib with --no-deps to prevent package backtracking
print("\n[3/3] 📦 Installing heartlib package...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", HEARTLIB_DIR, "--no-deps"],
    check=True,
)
print("✓ HeartMuLa library registered")

print("\n" + "=" * 65)
print("✅ Environment setup complete in ~25 seconds!")
print("=" * 65)

In [ ]:
# @title 📥 2. Download Checkpoints (High-Speed Downloader)
# @markdown Downloads HeartMuLa 3B and HeartCodec weights using huggingface_hub with automatic retry (one-time setup).
import os
import shutil
from huggingface_hub import snapshot_download

# Disable hf_transfer to prevent Rust binary / exit status 1 crashes
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

BASE_DIR = "/content/heartlib" if os.path.exists("/content/heartlib") else "."
CKPT_DIR = os.path.join(BASE_DIR, "ckpt")
os.makedirs(CKPT_DIR, exist_ok=True)
mula_dir = os.path.join(CKPT_DIR, "HeartMuLa-oss-3B")
codec_dir = os.path.join(CKPT_DIR, "HeartCodec-oss")
os.makedirs(mula_dir, exist_ok=True)
os.makedirs(codec_dir, exist_ok=True)

print("=" * 65)
print("📥 DOWNLOADING HEARTMULA MODELS (HUGGINGFACE HUB)")
print("=" * 65)

# 1. HeartMuLa-oss-3B weights
print("\n[1/3] 📦 Downloading HeartMuLa-oss-3B backbone...")
try:
    snapshot_download(
        repo_id="benjiaiplayground/HeartMuLa-oss-3B-bf16",
        local_dir=mula_dir,
        max_workers=4,
    )
    print("✓ HeartMuLa-oss-3B downloaded")
except Exception as e:
    print(f"⚠️ Primary source notice ({e}), using official HeartMuLa-oss-3B repository...")
    snapshot_download(
        repo_id="HeartMuLa/HeartMuLa-oss-3B",
        local_dir=mula_dir,
        max_workers=4,
    )
    print("✓ HeartMuLa-oss-3B downloaded from official repository")

# 2. HeartCodec-oss audio codec
print("\n[2/3] 📦 Downloading HeartCodec-oss codec...")
snapshot_download(
    repo_id="benjiaiplayground/HeartCodec-oss-bf16",
    local_dir=codec_dir,
    max_workers=4,
)
print("✓ HeartCodec-oss downloaded")

# 3. HeartMuLaGen config & tokenizer
print("\n[3/3] 📦 Downloading HeartMuLaGen config and tokenizer...")
snapshot_download(
    repo_id="HeartMuLa/HeartMuLaGen",
    local_dir=CKPT_DIR,
    max_workers=4,
)
print("✓ Config and Tokenizer downloaded")

# 4. Configure HeartCodec file naming and align state dict keys
codec_safetensors = os.path.join(codec_dir, "HeartCodec-oss-bf16.safetensors")
codec_target = os.path.join(codec_dir, "model.safetensors")
if os.path.exists(codec_safetensors) and not os.path.exists(codec_target):
    shutil.move(codec_safetensors, codec_target)

# Align HeartCodec checkpoint keys: add 'decoder.' prefix to ensure 100% weight loading
if os.path.exists(codec_target):
    try:
        from safetensors.torch import load_file, save_file
        weights = load_file(codec_target)
        needs_prefix = any(
            not k.startswith("decoder.") and (k.startswith("flow_matching.") or k.startswith("scalar_model."))
            for k in weights.keys()
        )
        if needs_prefix:
            print("\n🔧 Aligning HeartCodec keys with 'decoder.' prefix to ensure 100% weights load...")
            aligned_weights = {}
            for k, v in weights.items():
                if not k.startswith("decoder.") and (k.startswith("flow_matching.") or k.startswith("scalar_model.")):
                    aligned_weights[f"decoder.{k}"] = v
                else:
                    aligned_weights[k] = v
            save_file(aligned_weights, codec_target)
            print("✓ HeartCodec weights aligned! Eliminates missing keys and prevents silent audio.")
        else:
            print("✓ HeartCodec checkpoint keys verified and aligned.")
    except Exception as e:
        print(f"ℹ️ Codec alignment note: {e}")

print("\n" + "=" * 65)
print("✅ ALL MODELS DOWNLOADED AND CONFIGURED SUCCESSFULLY!")
print(f"📁 Checkpoints stored in: {CKPT_DIR}")
print("=" * 65)

In [ ]:
# @title 🎵 3. Launch HeartMuLa AIQUEST Music Studio (T4 FP16 Optimized)
# @markdown Launches the Gradio WebUI without embedded preview, streaming live inference logs continuously.
import os
import sys
import time
import gc
import io
import contextlib
import subprocess
from datetime import datetime
import torch
import gradio as gr
import psutil

# Prevent memory fragmentation on Tesla T4
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Ensure vector_quantize_pytorch is present even if Cell 1 was run previously
try:
    import vector_quantize_pytorch
except ImportError:
    print("📦 Installing required vector_quantize_pytorch package...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vector_quantize_pytorch>=1.14.0"], check=True)

# Ensure heartlib package is importable
BASE_DIR = "/content/heartlib" if os.path.exists("/content/heartlib") else "."
SRC_DIR = os.path.join(BASE_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
os.chdir(BASE_DIR)

CKPT_DIR = os.path.join(BASE_DIR, "ckpt")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Align HeartCodec weights if not already done (prevents silent audio)
codec_file = os.path.join(CKPT_DIR, "HeartCodec-oss", "model.safetensors")
if os.path.exists(codec_file):
    try:
        from safetensors.torch import load_file, save_file
        w = load_file(codec_file)
        if any(not k.startswith("decoder.") and (k.startswith("flow_matching.") or k.startswith("scalar_model.")) for k in w.keys()):
            print("🔧 Auto-aligning HeartCodec keys to eliminate silent audio...")
            aligned = {}
            for k, v in w.items():
                if not k.startswith("decoder.") and (k.startswith("flow_matching.") or k.startswith("scalar_model.")):
                    aligned[f"decoder.{k}"] = v
                else:
                    aligned[k] = v
            save_file(aligned, codec_file)
            print("✓ HeartCodec weights aligned successfully!")
    except Exception as e:
        print(f"ℹ️ Note on codec weights: {e}")

# Clean purge of any stale heartlib modules in this kernel session to guarantee
# 100% unified class identities and eliminate Python super(type, obj) reload mismatches
import sys
for _mod in list(sys.modules.keys()):
    if _mod == "heartlib" or _mod.startswith("heartlib."):
        del sys.modules[_mod]

from heartlib.heartmula.modeling_heartmula import HeartMuLa
from heartlib.heartcodec.modeling_heartcodec import HeartCodec
from heartlib import HeartMuLaGenPipeline

# Hardware math accelerations for Tesla T4 (Turing SM75)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")
if hasattr(torch.backends.cuda, "enable_mem_efficient_sdp"):
    torch.backends.cuda.enable_mem_efficient_sdp(True)
if hasattr(torch.backends.cuda, "enable_flash_sdp"):
    torch.backends.cuda.enable_flash_sdp(False)
if hasattr(torch.backends.cuda, "enable_math_sdp"):
    torch.backends.cuda.enable_math_sdp(True)

# Safe idempotent monkey-patch for HeartMuLa.setup_caches with dynamic sequence length bounding (saves 1.5+ GB VRAM)
def _patched_setup_caches(self, max_batch_size: int, max_seq_len: int = 2048):
    dtype = next(self.parameters()).dtype
    device = next(self.parameters()).device

    try:
        self.reset_caches()
    except Exception:
        pass

    with device:
        for m in self.modules():
            if hasattr(m, "rope_init"):
                try:
                    m.rope_init()
                except Exception:
                    pass

        # Constrain KV cache to actual sequence length needed (saves 1.5+ GB in High Fidelity Mode!)
        self.backbone.setup_caches(
            max_batch_size,
            dtype,
            decoder_max_seq_len=max_seq_len,
        )
        self.decoder.setup_caches(
            max_batch_size,
            dtype,
            decoder_max_seq_len=self.config.audio_num_codebooks,
        )

    # Create causal masks matching the bounded sequence length
    self.register_buffer(
        "backbone_causal_mask",
        torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool, device=device)),
    )
    self.register_buffer(
        "decoder_causal_mask",
        torch.tril(torch.ones(self.config.audio_num_codebooks, self.config.audio_num_codebooks, dtype=torch.bool, device=device)),
    )

HeartMuLa.setup_caches = _patched_setup_caches
HeartMuLa._is_setup_caches_patched = True

# High-speed postprocess: FP16 detokenization with adaptive flow-matching steps and soundfile I/O
def _fast_postprocess(self, model_outputs, save_path, num_steps=10, guidance_scale=1.25):
    frames = model_outputs["frames"].to(self.codec_device)
    codec = self.codec
    if next(codec.parameters()).dtype != self.codec_dtype:
        codec.to(dtype=self.codec_dtype)
    torch.cuda.empty_cache()
    gc.collect()
    wav = codec.detokenize(
        frames,
        num_steps=num_steps,
        guidance_scale=guidance_scale,
        disable_progress=True,
    )
    torch.cuda.empty_cache()
    import soundfile as sf
    wav_cpu = wav.to(torch.float32).cpu()
    if wav_cpu.dim() == 3:
        wav_cpu = wav_cpu[0]
    wav_np = wav_cpu.numpy()
    if wav_np.shape[0] in (1, 2) and wav_np.shape[1] > wav_np.shape[0]:
        wav_np = wav_np.T
    sf.write(save_path, wav_np, 48000)

# Hook postprocess on HeartMuLaGenPipeline
HeartMuLaGenPipeline.postprocess = _fast_postprocess

# Smooth, accurate progress tracker hook for HeartMuLaGenPipeline._forward
_CURRENT_PROGRESS = None

def _hooked_forward(self, model_inputs, max_audio_length_ms, temperature, topk, cfg_scale):
    prompt_tokens = model_inputs["tokens"].to(self.mula_device)
    prompt_tokens_mask = model_inputs["tokens_mask"].to(self.mula_device)
    continuous_segment = model_inputs["muq_embed"].to(self.mula_device)
    starts = model_inputs["muq_idx"]
    prompt_pos = model_inputs["pos"].to(self.mula_device)
    frames = []

    # Cache model reference outside the loop to avoid redundant property lookups
    mula_model = self.mula
    bs_size = 2 if cfg_scale != 1.0 else 1
    max_audio_frames = max_audio_length_ms // 80

    # Calculate exact max sequence length needed (prompt tokens + audio frames + safety buffer)
    prompt_len = prompt_tokens.shape[1] if prompt_tokens.dim() > 1 else 128
    needed_seq_len = min(2048, max(max_audio_frames + prompt_len + 64, 512))

    mula_model.setup_caches(bs_size, max_seq_len=needed_seq_len)
    with torch.autocast(device_type=self.mula_device.type, dtype=self.mula_dtype):
        curr_token = mula_model.generate_frame(
            tokens=prompt_tokens,
            tokens_mask=prompt_tokens_mask,
            input_pos=prompt_pos,
            temperature=temperature,
            topk=topk,
            cfg_scale=cfg_scale,
            continuous_segments=continuous_segment,
            starts=starts,
        )
    frames.append(curr_token[0:1,])

    def _pad_audio_token(token: torch.Tensor):
        padded_token = (
            torch.ones(
                (token.shape[0], self._parallel_number),
                device=token.device,
                dtype=torch.long,
            )
            * self.config.empty_id
        )
        padded_token[:, :-1] = token
        padded_token = padded_token.unsqueeze(1)
        padded_token_mask = torch.ones_like(
            padded_token, device=token.device, dtype=torch.bool
        )
        padded_token_mask[..., -1] = False
        return padded_token, padded_token_mask

    for i in range(max_audio_frames):
        curr_token, curr_token_mask = _pad_audio_token(curr_token)
        with torch.autocast(device_type=self.mula_device.type, dtype=self.mula_dtype):
            curr_token = mula_model.generate_frame(
                tokens=curr_token,
                tokens_mask=curr_token_mask,
                input_pos=prompt_pos[..., -1:] + i + 1,
                temperature=temperature,
                topk=topk,
                cfg_scale=cfg_scale,
                continuous_segments=None,
                starts=None,
            )
        if torch.any(curr_token[0:1, :] >= self.config.audio_eos_id):
            break
        frames.append(curr_token[0:1,])

        # Smooth, monotonically increasing progress updates in Gradio UI
        global _CURRENT_PROGRESS
        if _CURRENT_PROGRESS is not None and (i % 5 == 0 or i == max_audio_frames - 1):
            ratio = (i + 1) / max_audio_frames
            cur_sec = (i + 1) * 0.08
            total_sec = max_audio_frames * 0.08
            _CURRENT_PROGRESS(
                0.05 + 0.85 * ratio,
                desc=f"🎹 Generating Frame {i+1}/{max_audio_frames} ({cur_sec:.1f}s / {total_sec:.1f}s audio)..."
            )

    frames = torch.stack(frames).permute(1, 2, 0).squeeze(0)
    # Purge KV cache buffers and causal masks to free 1.1+ GB VRAM for HeartCodec
    for m in mula_model.modules():
        if hasattr(m, "kv_cache"):
            m.kv_cache = None
    if hasattr(mula_model, "backbone_causal_mask"):
        mula_model.backbone_causal_mask = None
    if hasattr(mula_model, "decoder_causal_mask"):
        mula_model.decoder_causal_mask = None
    gc.collect()
    torch.cuda.empty_cache()
    return {"frames": frames}

HeartMuLaGenPipeline._forward = _hooked_forward

# Global in-memory pipeline cache
PIPELINE = None

def get_pipeline():
    """Initializes pipeline with dual FP16 resident in GPU VRAM (zero host RAM usage)."""
    global PIPELINE
    if PIPELINE is not None:
        return PIPELINE

    print("⏳ Initializing HeartMuLa 3B Pipeline (T4 Dual FP16 Acceleration)...")
    t0 = time.time()
    PIPELINE = HeartMuLaGenPipeline.from_pretrained(
        CKPT_DIR,
        device={
            "mula": torch.device("cuda"),
            "codec": torch.device("cuda"),
        },
        dtype={
            "mula": torch.float16,   # ⚡ Native FP16 Tensor Cores on Tesla T4 (~5.8 GB VRAM)
            "codec": torch.float16,  # ⚡ FP16 FlowMatching (~3.1 GB VRAM)
        },
        version="3B",
        lazy_load=False,             # ⚡ Both models resident in VRAM (~8.9 GB / 15.0 GB total, 0s disk reload)
    )

    # Ensure both models are strictly in FP16 to guarantee ~8.9 GB total VRAM footprint
    if hasattr(PIPELINE, "_mula") and PIPELINE._mula is not None:
        PIPELINE._mula.to(dtype=torch.float16)
    if hasattr(PIPELINE, "_codec") and PIPELINE._codec is not None:
        PIPELINE._codec.to(dtype=torch.float16)

    print(f"✅ Pipeline initialized in {time.time() - t0:.2f}s!")
    print(f"📊 VRAM Allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GB / {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
    return PIPELINE

def generate_music_studio(
    lyrics: str,
    tags: str,
    duration: int,
    generation_mode: str,
    temperature: float,
    topk: int,
    cfg_scale: float,
    progress=gr.Progress(),
):
    """Generate music with live terminal logging, smooth progress bar, and memory cleanup."""
    if not lyrics.strip():
        return None, "⚠️ Please provide lyrics for the song."
    if not tags.strip():
        return None, "⚠️ Please provide style tags (e.g., piano, pop, acoustic)."

    # Fast draft mode overrides CFG scale to 1.0 (batch size = 1, 2x faster token throughput)
    is_fast_mode = "Fast" in generation_mode
    effective_cfg = 1.0 if is_fast_mode else cfg_scale
    codec_steps = 8 if is_fast_mode else 10
    codec_cfg = 1.0 if is_fast_mode else 1.25

    # Clear lingering GPU memory before launching generation
    gc.collect()
    torch.cuda.empty_cache()

    duration_ms = int(duration * 1000)
    total_frames = duration_ms // 80

    print("\n" + "=" * 65)
    print(f"🎵 Starting Music Generation: {duration}s ({total_frames} frames)")
    print(f"🎚️ Settings: Mode={generation_mode} | Mula CFG={effective_cfg} | Temp={temperature} | Top-K={topk}")
    print(f"⚡ Detokenizer: FlowMatching {codec_steps} steps (CFG={codec_cfg}, FP16 Tensor Cores)")
    print(f"📝 Style: {tags.strip()[:60]}...")
    print(f"📊 Starting VRAM Allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")
    print("=" * 65)

    global _CURRENT_PROGRESS
    _CURRENT_PROGRESS = progress
    progress(0.02, desc=f"🎵 Preparing HeartMuLa pipeline ({total_frames} frames)...")

    try:
        start_time = time.time()
        pipe = get_pipeline()

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = os.path.join(OUTPUT_DIR, f"music_{timestamp}.wav")

        print("⏳ Step 1/2: Generating music tokens with HeartMuLa 3B...")
        t_start_tokens = time.time()

        with torch.inference_mode():
            # Run preprocess and forward
            model_inputs = pipe.preprocess(
                {"lyrics": lyrics, "tags": tags},
                cfg_scale=float(effective_cfg)
            )
            model_outputs = pipe._forward(
                model_inputs,
                max_audio_length_ms=duration_ms,
                temperature=float(temperature),
                topk=int(topk),
                cfg_scale=float(effective_cfg),
            )
            t_tokens = time.time() - t_start_tokens
            print(f"✓ Token generation complete in {t_tokens:.1f}s ({total_frames/t_tokens:.1f} fps)")

            # Purge lingering cache and check available VRAM before detokenization
            free_vram_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / (1024**3)
            print(f"⚡ VRAM Headroom for HeartCodec: {free_vram_gb:.2f} GB free")

            # Step 2: Audio Detokenization
            progress(0.92, desc=f"🎧 Step 2/2: Detokenizing audio with HeartCodec ({codec_steps} steps, FP16)...")
            print(f"⏳ Step 2/2: Detokenizing audio waveform with HeartCodec ({codec_steps} steps, FP16 Tensor Cores)...")
            t_start_codec = time.time()
            pipe.postprocess(
                model_outputs,
                save_path=output_file,
                num_steps=codec_steps,
                guidance_scale=codec_cfg,
            )
            t_codec = time.time() - t_start_codec
            print(f"✓ Audio decoded in {t_codec:.1f}s (was 94.1s in FP32)!")

        elapsed = time.time() - start_time
        fps = total_frames / elapsed if elapsed > 0 else 0
        rtf = elapsed / duration if duration > 0 else 0

        progress(1.0, desc="✅ Music generation complete!")
        print(f"\n✓ Music generation finished in {elapsed:.1f}s!")
        print(f"✓ Output saved: {output_file}")
        print(f"⚡ Generation Speed: {fps:.1f} frames/sec (RTF: {rtf:.2f}x)")
        print(f"📊 Peak VRAM Allocated: {torch.cuda.max_memory_allocated() / (1024**3):.2f} GB")
        print("=" * 65 + "\n")

        stats = f"""### ✅ Generation Complete!
- 🎵 **Audio Duration:** {duration}s ({duration/60:.1f} min)
- ⏱️ **Elapsed Time:** {elapsed:.1f}s ({elapsed/60:.2f} min)
- ⚡ **Generation Speed:** {fps:.1f} frames/sec (RTF: {rtf:.2f}x)
- 🎚️ **Settings:** Mode={generation_mode} | CFG={effective_cfg} | Codec={codec_steps} steps
- 💾 **Precision:** HeartMuLa FP16 | HeartCodec FP16 (Tesla T4 Tensor Cores)
- 🚀 **Memory:** Pure VRAM-Resident Dual FP16 (0s disk reload, zero host RAM overhead)
"""
        return output_file, stats

    except Exception as e:
        print(f"\n❌ Generation Error: {str(e)}\n")
        return None, f"❌ Generation Error: {str(e)}"

    finally:
        _CURRENT_PROGRESS = None
        gc.collect()
        torch.cuda.empty_cache()

# Custom AIQUEST Academy CSS
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1100px !important; margin: auto !important; }
.brand-header {
    text-align: center;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    padding: 30px;
    border-radius: 15px;
    margin-bottom: 20px;
    box-shadow: 0 10px 25px rgba(102,126,234,0.3);
}
.brand-title { color: white; font-size: 2.2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.9); font-size: 1.05em; margin: 0 0 16px 0; text-align: center; }
.social-buttons { display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }
.social-btn {
    padding: 10px 24px;
    border-radius: 8px;
    font-weight: 700;
    font-size: 14px;
    text-decoration: none;
    display: inline-block;
    color: white !important;
    transition: all 0.3s;
    box-shadow: 0 4px 12px rgba(0,0,0,0.2);
}
.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }
.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }
.x-btn { background: linear-gradient(135deg, #000000 0%, #333333 100%); }
button.primary, #gen-btn {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    color: white !important;
    font-weight: 600 !important;
    border-radius: 12px !important;
}
#stop-btn {
    background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important;
    color: white !important;
    font-weight: 600 !important;
    border-radius: 12px !important;
}
#clear-btn {
    background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important;
    color: white !important;
    font-weight: 600 !important;
    border-radius: 12px !important;
}
.footer {
    text-align: center;
    padding: 20px;
    margin-top: 30px;
    border-top: 2px solid #e5e7eb;
    color: #6b7280;
}
"""

with gr.Blocks(title="🎵 HeartMuLa 3B - AIQUEST Academy") as demo:

    # AIQUEST Academy Header
    gr.HTML("""
    <div class="brand-header">
        <h1 class="brand-title">🎵 HeartMuLa 3B - Standalone Music Generator</h1>
        <p class="brand-subtitle">Google Colab T4 GPU Edition - Created by <strong>AIQUEST Academy</strong></p>
        <div class="social-buttons">
            <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn youtube-btn">🔴 Subscribe on YouTube</a>
            <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">🐦 Follow on X</a>
        </div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=5):
            gr.Markdown("### 📝 Song Lyrics & Style Tags")
            lyrics_input = gr.Textbox(
                label="Lyrics (Use structural tags: [Intro], [Verse], [Chorus], [Bridge], [Outro])",
                placeholder="[Intro]\nSoft piano melody begins\n\n[Verse]\nThe sun creeps in across the floor\nAnother morning just like this\n\n[Chorus]\nEvery day the light returns\nEvery day the fire burns",
                lines=8,
                value="""[Intro]
Soft acoustic melody begins

[Verse]
The morning light streams through the glass
Another quiet day to pass
I hear the city start to wake
A brand new journey we can make

[Chorus]
Hold on to the dream alive
Through the darkest night we strive
Rising up into the blue
Everything is clear and true""",
            )

            tags_input = gr.Textbox(
                label="Style Tags (Comma-separated genre, instruments, and mood)",
                placeholder="piano, acoustic, calm, morning, pop",
                value="acoustic, guitar, calm, melodic, folk pop",
            )

            gr.Markdown("### ⚡ Generation & Performance Controls")
            mode_radio = gr.Radio(
                choices=["⚡ Fast Draft Mode (CFG = 1.0, 2x Faster)", "🎼 High Fidelity Mode (CFG = 1.5)"],
                value="⚡ Fast Draft Mode (CFG = 1.0, 2x Faster)",
                label="Performance Mode",
                info="Fast Draft Mode uses batch size 1 to cut generation time in half on T4 GPU.",
            )

            duration_slider = gr.Slider(
                minimum=10, maximum=120, value=60, step=5,
                label="Audio Duration (Seconds)",
                info="⚡ Default: 60s (~2.0-2.3 min generation). Use 15s-30s for faster draft previews.",
            )

            with gr.Accordion("⚙️ Advanced Hyperparameters (Temperature, Top-K, CFG)", open=False):
                with gr.Row():
                    temperature_slider = gr.Slider(
                        0.5, 1.5, value=1.0, step=0.05,
                        label="Temperature",
                        info="Higher = more creative/experimental",
                    )
                    topk_slider = gr.Slider(
                        10, 100, value=50, step=5,
                        label="Top-K",
                        info="Vocabulary sampling filter",
                    )
                cfg_scale_slider = gr.Slider(
                    1.0, 3.0, value=1.5, step=0.1,
                    label="Custom CFG Scale (Used in High Fidelity Mode)",
                    info="Style adherence strength",
                )

            # 3 Standard AIQUEST Buttons
            with gr.Row():
                gen_btn = gr.Button("🎵 Generate Music", variant="primary", size="lg", elem_id="gen-btn")
                stop_btn = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")

        with gr.Column(scale=4):
            gr.Markdown("### 🎧 Generated Audio")
            output_audio = gr.Audio(label="Music Player", type="filepath", interactive=False)
            stats_output = gr.Markdown("ℹ️ Select a preset or enter lyrics, then click **Generate Music**.")

            gr.Markdown("""
            ### 💡 T4 GPU Speed & Duration Guide
            - **⚡ 60s Full Song (Default):** ~2.0-2.3 min total generation time (Fast Mode).
            - **⚡ 30s Audio:** ~60-75s total generation time (Fast Mode).
            - **⚡ 15s Lightning Preview:** ~32-38s total generation time (Fast Mode).
            - **🚀 Pure VRAM-Resident:** Both models stay loaded directly on GPU (0s reload, zero host RAM overhead).
            - **🔥 Dual FP16 Tensor Cores:** Blazing fast detokenization (~8-12s instead of 94s!).
            """)

    # Preset Examples
    gr.Markdown("### 🎼 Curated Presets (Click to Load)")
    gr.Examples(
        examples=[
            [
                "[Intro]\nFast acoustic guitar strumming\n\n[Chorus]\nChase the morning light\nEverything is feeling right\nRising up so high",
                "acoustic guitar, upbeat, energetic, folk pop, morning",
                15,
                "⚡ Fast Draft Mode (CFG = 1.0, 2x Faster)",
                1.0,
                50,
                1.5,
            ],
            [
                "[Verse]\nNeon lights paint the street\nCity pulse beneath my feet\n\n[Chorus]\nAlive tonight, feeling right\nDancing under violet skies",
                "synthwave, electronic, upbeat, 80s drums, energetic",
                30,
                "⚡ Fast Draft Mode (CFG = 1.0, 2x Faster)",
                1.0,
                50,
                1.5,
            ],
            [
                "[Verse]\nQuiet morning light\nSoft and bright\nPeaceful moments here\n\n[Chorus]\nStay a while with me\nLet the world drift away",
                "acoustic, calm, gentle guitar, relaxing, folk",
                30,
                "⚡ Fast Draft Mode (CFG = 1.0, 2x Faster)",
                0.9,
                40,
                1.5,
            ],
            [
                "[Verse]\nThunder rolls across the sky\nLightning flashes way up high\n\n[Chorus]\nStorm is here, feel the power\nRocking through the midnight hour",
                "hard rock, heavy electric guitar, energetic drums",
                30,
                "🎼 High Fidelity Mode (CFG = 1.5)",
                1.05,
                60,
                1.8,
            ],
            [
                "[Intro]\nJazz piano chords\n\n[Verse]\nSmooth nights in the lounge\nSax melody all around\n\n[Chorus]\nLost in rhythm, found in sound",
                "jazz, lounge, saxophone, upright bass, smooth piano",
                30,
                "🎼 High Fidelity Mode (CFG = 1.5)",
                1.0,
                50,
                1.5,
            ],
        ],
        inputs=[
            lyrics_input,
            tags_input,
            duration_slider,
            mode_radio,
            temperature_slider,
            topk_slider,
            cfg_scale_slider,
        ],
    )

    # Footer
    gr.HTML("""
    <div class="footer">
        <p>HeartMuLa 3B Music Generator - Powered by <strong>HeartMuLa Team</strong> & <strong>benjiaiplayground</strong>.</p>
        <p>Optimized for Google Colab Free Tier by <strong>AIQUEST Academy</strong>.</p>
    </div>
    """)

    # Button Event Handlers
    gen_event = gen_btn.click(
        fn=generate_music_studio,
        inputs=[
            lyrics_input,
            tags_input,
            duration_slider,
            mode_radio,
            temperature_slider,
            topk_slider,
            cfg_scale_slider,
        ],
        outputs=[output_audio, stats_output],
    )
    stop_btn.click(fn=None, cancels=[gen_event])
    clear_btn.click(
        fn=lambda: (None, "ℹ️ Cleared. Ready to generate.", "", ""),
        outputs=[output_audio, stats_output, lyrics_input, tags_input],
    )

print("\n" + "=" * 65)
print("🎵 Launching HeartMuLa AIQUEST Music Studio WebUI...")
print("=" * 65)

# Launch without notebook iframe preview and suppress promotional messages
_suppress_buffer = io.StringIO()
with contextlib.redirect_stdout(_suppress_buffer):
    app, local_url, share_url = demo.queue(max_size=5).launch(
        theme=gr.themes.Soft(),
        css=custom_css,
        share=True,
        inline=False,
        debug=False,
        prevent_thread_lock=True,
    )

print(f"\n🌟 AIQUEST Music Studio is LIVE!")
print(f"🔗 Public URL: {share_url}")
print(f"🔗 Local URL:  {local_url}")
print("=" * 65)
print("⚡ WebUI active! Real-time inference logs will stream below.\n")

# Keep the cell execution active and continuously stream all inference logs to console
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Session terminated by user.")